# 경기도 농업기상 전처리

위에서부터 **Shift+Enter**로 한 셀씩 실행하세요. 처음 실행할 때는 전체 CSV를 읽고 정제 파일을 만드는 셀이 가장 오래 걸립니다. 확인만 다시 할 때는 설정 셀과 아래의 **확인할 관측소와 날짜** 셀을 실행하면 저장된 결과를 볼 수 있습니다.

01에서는 원본의 경기도 관측소를 모두 정제해 `weather_clean.csv`를 만듭니다. 02에서는 08:00~17:00 작업시간으로 묶어 `weather_workday_features.csv`를 만듭니다.

In [ ]:
import pickle
from math import isfinite
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

# 집계용 시범 기준이며 작업별 검증 후 변경할 수 있습니다.
heat_threshold_c = 33.0
wind_threshold_mps = 10.0
if not isfinite(heat_threshold_c) or not isfinite(wind_threshold_mps) or wind_threshold_mps < 0:
    raise ValueError("온도 기준은 유한한 값, 풍속 기준은 유한한 0 이상의 값이어야 합니다.")

cwd = Path.cwd().resolve()
project = next(
    (root for root in (cwd, *cwd.parents) if (root / "worker/code/preprocess_weather.py").is_file()),
    None,
)
if project is None:
    raise FileNotFoundError("WON데이 저장소 안에서 노트북을 실행하세요.")
data_dir = project / "data"
raw_dir = data_dir / "gyeonggi-2016-2026-csv"
daily_rain_path = data_dir / "gyeonggi_daily_rainfall.csv"
output_dir = data_dir / "gyeonggi_processed"
clean_path = output_dir / "weather_clean.csv"
workday_path = output_dir / "weather_workday_features.csv"
raw_files = sorted(raw_dir.glob("*.csv"))
if not raw_files:
    raise FileNotFoundError(f"원본 CSV 폴더를 찾을 수 없습니다. {raw_dir}")

print(f"원본 CSV {len(raw_files):,}개")
print(f"일강수량  {daily_rain_path if daily_rain_path.is_file() else '파일 없음'}")
print(f"결과 폴더  {output_dir}")

## 01 기상 데이터 확인 및 정제

먼저 원본에서 열 이름과 기록 예시를 확인합니다. `강수량`은 해당 날짜의 누적값입니다. 작업에 필요한 관측소 코드·지점명·시각·기온·습도·풍속·강수량을 사용합니다.

In [ ]:
columns = {
    "station_code": "station_code",
    "지점명": "station_name",
    "날짜": "observed_at",
    "기온( ℃ )": "temperature_c",
    "습도( % )": "humidity_pct",
    "풍속( m/s )": "wind_mps",
    "강수량( mm )": "daily_cumulative_rain_mm",
}
display(pd.read_csv(raw_files[0], usecols=list(columns), nrows=5))

아래 셀은 `X`와 비어 있는 값을 결측으로 정리합니다. 원본의 `24:00`은 다음 날 `00:00`으로 표시합니다. 마지막 10분 구간은 같은 관측소·날짜의 국립농업과학원 일강수량에서 `23:50` 누적값을 빼서 계산합니다. 이 두 값이 있으면 `24:00` 관측 행이 빠져도 강수량 집계에 반영합니다. 원본 누적값은 정제 파일에 그대로 보관합니다. 일강수량이 없고 `24:00`의 원본 누적값이 0이면 마지막 구간은 결측으로 남깁니다. 다음 날 `00:10`의 첫 누적값은 그날 첫 10분 구간의 강수량입니다.

3시간과 12시간 값은 하루 전체에서 10분씩 이동해 계산합니다. 창 안에 필요한 18개 또는 72개 강수 구간이 모두 있을 때 값이 표시됩니다.

In [ ]:
measurement_columns = ["temperature_c", "humidity_pct", "wind_mps", "daily_cumulative_rain_mm"]
daily_totals = {}
if daily_rain_path.is_file():
    daily_keys = ["station_code", "work_date"]
    daily = pd.read_csv(daily_rain_path, usecols=[*daily_keys, "daily_rain_mm"], dtype={"station_code": str})
    daily["work_date"] = pd.to_datetime(daily["work_date"], format="%Y-%m-%d")
    totals = pd.to_numeric(daily["daily_rain_mm"], errors="coerce")
    daily = daily.loc[totals.ge(0) & totals.lt(float("inf"))].assign(daily_rain_mm=totals).drop_duplicates()
    if daily.duplicated(daily_keys).any():
        raise ValueError("관측소와 날짜가 같고 일강수량이 다른 행이 있습니다.")
    for station, group in daily.groupby("station_code"):
        daily_totals[station] = group.set_index("work_date")["daily_rain_mm"]

def read_raw(path):
    frame = pd.read_csv(
        path, usecols=["도명", *columns],
        dtype=str, keep_default_na=False, encoding="utf-8-sig",
    )
    frame = frame.loc[frame["도명"].str.strip().eq("경기도")].drop(columns="도명")
    return frame.rename(columns=columns)

def regular_rain_grid(frame, station):
    source = frame.set_index("observed_at")["daily_cumulative_rain_mm"]
    rain = source.asfreq("10min")
    end_times = rain.index[rain.index == rain.index.normalize()]
    end_values = rain.loc[end_times].where(rain.loc[end_times].gt(0))
    if station in daily_totals:
        official = daily_totals[station].reindex(end_times - pd.Timedelta(days=1))
        official.index = end_times
        end_values = official.combine_first(end_values)
    rain.loc[end_times] = end_values
    increments = rain.diff()
    first_interval = (rain.index.hour == 0) & (rain.index.minute == 10)
    increments.loc[first_interval] = rain.loc[first_interval]
    return increments.where(increments.ge(0))

def clean_station(parts):
    frame = pd.concat(parts, ignore_index=True)
    timestamp = frame["observed_at"].str.strip()
    midnight = timestamp.str.endswith(" 24:00")
    corrected = timestamp.where(~midnight, timestamp.str.slice(stop=-5) + "00:00")
    frame["observed_at"] = (
        pd.to_datetime(corrected, format="%Y-%m-%d %H:%M") +
        pd.to_timedelta(midnight.astype(int), unit="D")
    )
    if frame["observed_at"].dt.minute.mod(10).ne(0).any():
        raise ValueError("10분 간격에 맞지 않는 관측 시각이 있습니다.")

    for column in measurement_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame[column] = frame[column].where(frame[column].abs().lt(float("inf")))
    frame["humidity_pct"] = frame["humidity_pct"].where(frame["humidity_pct"].between(0, 100))
    for column in ["wind_mps", "daily_cumulative_rain_mm"]:
        frame[column] = frame[column].where(frame[column].ge(0))

    frame = frame.drop_duplicates(subset=["observed_at", *measurement_columns])
    if frame["observed_at"].duplicated().any():
        station = frame["station_code"].iloc[0]
        raise ValueError(f"관측값이 서로 다른 중복 시각이 있습니다. 관측소 {station}")

    frame = frame.sort_values("observed_at").reset_index(drop=True)
    rain = regular_rain_grid(frame, frame["station_code"].iloc[0])
    frame["rain_10min_mm"] = frame["observed_at"].map(rain).round(3)
    for hours in (3, 12):
        count = hours * 6
        rolling = rain.rolling(count, min_periods=count).sum().round(3)
        frame[f"rain_{hours}h_mm"] = frame["observed_at"].map(rolling)
    return frame

In [ ]:
def read_parts(path):
    parts = []
    with path.open("rb") as file:
        while file.peek(1):
            parts.append(pickle.load(file))
    return parts

# 여러 관측소의 원본을 한 번 읽고, 정제는 관측소 한 곳씩 진행합니다.
output_dir.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix="weather-", dir=output_dir) as temporary:
    temp = Path(temporary)
    station_parts = {}
    batch_size = 20
    for start in range(0, len(raw_files), batch_size):
        batch = pd.concat(
            [read_raw(path) for path in raw_files[start:start + batch_size]], ignore_index=True
        )
        for station, group in batch.groupby("station_code"):
            if station not in station_parts:
                station_parts[station] = temp / f"{len(station_parts)}.pkl"
            with station_parts[station].open("ab") as file:
                pickle.dump(group, file, protocol=pickle.HIGHEST_PROTOCOL)
        if start % (batch_size * 5) == 0:
            print(f"읽은 원본 {min(start + batch_size, len(raw_files)):,}/{len(raw_files):,}개")
    if not station_parts:
        raise ValueError("경기도 관측소가 없습니다.")

    staged_clean = temp / clean_path.name
    clean_summary = []
    with staged_clean.open("w", newline="", encoding="utf-8-sig") as output:
        for station in sorted(station_parts):
            clean = clean_station(read_parts(station_parts[station]))
            clean.to_csv(output, index=False, header=not clean_summary)
            clean_summary.append({
                "station_code": station,
                "station_name": clean["station_name"].iloc[-1],
                "observations": len(clean),
                "missing_rain": int(clean["rain_10min_mm"].isna().sum()),
            })
    staged_clean.replace(clean_path)

station_summary = pd.DataFrame(clean_summary)
print(f"정제 완료  관측소 {len(station_summary):,}곳 / 관측 {station_summary.observations.sum():,}행")
display(station_summary)

## 02 농작업 기준 데이터셋 생성

관측소별로 날짜당 한 줄을 만듭니다. 작업시간은 08:00~17:00이며, 첫 10분 구간은 08:10에 끝나고 마지막은 17:00에 끝납니다. 작업시간은 54개 구간, 작업 전 24시간은 144개 구간이 필요합니다. 필요한 값이 부족한 항목은 빈칸으로 저장됩니다.

고온·강풍 지속시간은 설정 셀에서 지정한 기준 이상인 10분 구간 수에 10분을 곱합니다. 하루 중 기준을 충족한 구간을 합산하고, 해당 변수의 작업 중 관측이 54개 미만이면 빈칸으로 저장합니다.

In [ ]:
def iter_clean_stations():
    # 정제 파일은 관측소 코드 순으로 저장되므로 한 번에 한 관측소만 읽습니다.
    pending = []
    current = None
    for chunk in pd.read_csv(
        clean_path, chunksize=250_000, parse_dates=["observed_at"], dtype={"station_code": str}
    ):
        for station, rows in chunk.groupby("station_code", sort=False):
            if pending and station != current:
                yield current, pd.concat(pending, ignore_index=True)
                pending = []
            pending.append(rows)
            current = station
    if pending:
        yield current, pd.concat(pending, ignore_index=True)

def summarize_workdays(station, frame):
    weather = frame.set_index("observed_at")
    observation_dates = (weather.index - pd.Timedelta(minutes=1)).normalize()
    dates = pd.date_range(observation_dates.min(), observation_dates.max(), freq="D")
    daily = pd.DataFrame({"station_code": station}, index=dates)
    minutes = weather.index.hour * 60 + weather.index.minute
    work = weather.loc[(minutes > 8 * 60) & (minutes <= 17 * 60)]
    groups = work.groupby(work.index.normalize())

    daily["work_rain_mm"] = groups["rain_10min_mm"].sum(min_count=54).round(3)
    rain = regular_rain_grid(frame, station)
    starts = dates + pd.Timedelta(hours=8)
    daily["prior_24h_rain_mm"] = rain.rolling(144, min_periods=144).sum().reindex(starts).to_numpy().round(3)
    for column, source, operation in [
        ("work_max_temp_c", "temperature_c", "max"),
        ("work_mean_humidity_pct", "humidity_pct", "mean"),
        ("work_max_wind_mps", "wind_mps", "max"),
    ]:
        daily[column] = groups[source].agg(operation).where(groups[source].count().eq(54)).round(3)
    daily["work_rain_minutes"] = (
        work["rain_10min_mm"].gt(0).groupby(work.index.normalize()).sum() * 10
    ).where(groups["rain_10min_mm"].count().eq(54)).astype("Int64")
    daily["work_observation_count"] = groups.size().reindex(dates, fill_value=0)
    daily["work_rain_interval_count"] = groups["rain_10min_mm"].count().reindex(dates, fill_value=0)
    prior_count = rain.rolling(144, min_periods=1).count().reindex(starts).fillna(0)
    daily["prior_rain_interval_count"] = prior_count.to_numpy(dtype=int)
    for column, source, threshold in [
        ("work_heat_minutes", "temperature_c", heat_threshold_c),
        ("work_wind_minutes", "wind_mps", wind_threshold_mps),
    ]:
        duration = work[source].ge(threshold).groupby(work.index.normalize()).sum() * 10
        daily[column] = duration.where(groups[source].count().eq(54)).astype("Int64")
    daily["heat_threshold_c"] = heat_threshold_c
    daily["wind_threshold_mps"] = wind_threshold_mps
    return daily.rename_axis("work_date").reset_index()

workdays = pd.concat(
    [summarize_workdays(station, weather) for station, weather in iter_clean_stations()],
    ignore_index=True,
)
with TemporaryDirectory(prefix="weather-", dir=output_dir) as temporary:
    staged_workdays = Path(temporary) / workday_path.name
    workdays.to_csv(staged_workdays, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    staged_workdays.replace(workday_path)

print(f"집계 완료  관측소 {workdays.station_code.nunique():,}곳 / 관측소·날짜 {len(workdays):,}행")
display(workdays.head())

## 확인할 관측소와 날짜

관측소 코드와 날짜를 바꾼 뒤 아래 셀을 다시 실행하세요. 10분 표에는 작업시간 밖의 관측도 포함됩니다. 아래 예시 날짜의 가평읍 작업 중 강수량은 **24.5mm**입니다. 시간별 표의 `rain_3h_mm`과 `rain_12h_mm`에서 비가 어느 시간대에 집중됐는지 확인할 수 있습니다.

In [ ]:
station_code = "477802A001"
check_date = "2024-07-08"

rain_columns = ["daily_cumulative_rain_mm", "rain_10min_mm", "rain_3h_mm", "rain_12h_mm"]
start = pd.Timestamp(check_date)
end = start + pd.Timedelta(days=1)
pieces = []
for chunk in pd.read_csv(
    clean_path, chunksize=250_000, parse_dates=["observed_at"], dtype={"station_code": str}
):
    selection = chunk["station_code"].eq(station_code) & chunk["observed_at"].between(
        start, end, inclusive="right"
    )
    pieces.append(chunk.loc[selection, ["observed_at", "station_name", *rain_columns]])

day_weather = pd.concat(pieces, ignore_index=True)
if day_weather.empty:
    print("선택한 관측소와 날짜의 관측이 없습니다.")
else:
    print(f"{day_weather.station_name.iloc[0]}  {check_date}")
    display(day_weather[rain_columns].max().rename("해당 날짜 최댓값(mm)").to_frame())
    with pd.option_context("display.max_rows", 144):
        display(day_weather)

daily_result = pd.read_csv(workday_path, dtype={"station_code": str})
display(daily_result.loc[
    daily_result["station_code"].eq(station_code) & daily_result["work_date"].eq(check_date)
])